# ComfyUI on Radeon Cloud

This notebook starts ComfyUI in your instance and opens it in your browser. The image has ComfyUI and PyTorch for ROCm, and no models: you download the ones you want.

Run the cells from top to bottom. See [README.md](README.md) for which workflows fit a 1-GPU instance.

The earlier version of this Gallery template was the [ComfyUI + LTX-2.3 workshop notebook](https://github.com/Vivicai1005/notebook/tree/main/comfyui_ltx_notebook) by Vivicai1005.

## 1. Check the GPU

In [ ]:
import torch
print("GPU available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0))
print("GPU memory: %.0f GB" % (torch.cuda.get_device_properties(0).total_memory / 1e9))

## 2. Keep models in /workspace

ComfyUI reads models from `/opt/ComfyUI/models`, which is lost when the instance is deleted. This cell creates `/workspace/ComfyUI-data/models` and tells ComfyUI to look there too. If you chose **Persistent /workspace** at launch, downloaded models stay for your next instance.

In [ ]:
from pathlib import Path
root = Path("/workspace/ComfyUI-data/models")
for sub in ["checkpoints", "diffusion_models", "text_encoders", "loras", "vae", "latent_upscale_models"]:
    (root / sub).mkdir(parents=True, exist_ok=True)
cfg = Path("/workspace/ComfyUI-data/extra_model_paths.yaml")
cfg.write_text(
    "workspace:\n"
    "    base_path: /workspace/ComfyUI-data/models\n"
    "    checkpoints: checkpoints\n"
    "    diffusion_models: diffusion_models\n"
    "    text_encoders: text_encoders\n"
    "    loras: loras\n"
    "    vae: vae\n"
    "    latent_upscale_models: latent_upscale_models\n"
)
print("Models folder:", root)

## 3. Start ComfyUI

ComfyUI runs in the background on port 8188. Its log is in `/workspace/ComfyUI-data/comfyui.log`.

In [ ]:
import subprocess, time, urllib.request
log = open("/workspace/ComfyUI-data/comfyui.log", "a")
subprocess.Popen(
    ["comfyui-start", "--extra-model-paths-config", "/workspace/ComfyUI-data/extra_model_paths.yaml"],
    stdout=log, stderr=subprocess.STDOUT, start_new_session=True,
)
for _ in range(120):
    try:
        urllib.request.urlopen("http://127.0.0.1:8188/", timeout=2)
        print("ComfyUI is running on port 8188")
        break
    except Exception:
        time.sleep(2)
else:
    print("ComfyUI did not start; see /workspace/ComfyUI-data/comfyui.log")

## 4. Open ComfyUI in your browser

Install the tunnel tool once, then expose port 8188. The second command prints an `https://` address: open it in your browser.

If `rc-tunnel` is not found, open a new terminal (File > New > Terminal) and run `rc-tunnel expose --port 8188` there.

In [ ]:
!sh /var/run/secrets/frp-self-service/install
!$HOME/.local/bin/rc-tunnel expose --port 8188

## 5. Add models

This image has no models. Download the ones your workflow needs and put them in the matching folder under `/workspace/ComfyUI-data/models/` (kept with **Persistent /workspace**) or `/opt/ComfyUI/models/` (deleted with the instance): `checkpoints/`, `diffusion_models/`, `text_encoders/`, `loras/`, `vae/`, `latent_upscale_models/`. Then refresh the ComfyUI page.

A 1-GPU instance has 55 GB of system memory, and ComfyUI loads model files into it first. Check the file sizes before you download; see [README.md](README.md).

## 6. When you are done

Delete the instance from the console when you no longer need it. Files outside `/workspace` are deleted with it.